In [64]:
!pip install transformers accelerate optimum
!pip install autoawq
!pip install -q torch bitsandbytes rich psutil
print("all dependencies installed")

all dependencies installed


In [65]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import re
import time
import textwrap



In [66]:
model_name = "Qwen/Qwen2.5-3B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print(f"Loading {model_name} in 4-bit...")

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

Loading Qwen/Qwen2.5-3B-Instruct in 4-bit...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [67]:
doc = []
for i in range(50000):
    doc.append(f"Line {i}: This is a synthetic long context line. Index={i}.")
CONTEXT = "\n".join(doc)

print("Characters:", len(CONTEXT))
print("Preview:\n", CONTEXT[:200])


Characters: 3177779
Preview:
 Line 0: This is a synthetic long context line. Index=0.
Line 1: This is a synthetic long context line. Index=1.
Line 2: This is a synthetic long context line. Index=2.
Line 3: This is a synthetic long


In [147]:
TOOL_REGEX = re.compile(r'^TOOL:\s*([a-zA-Z_]+)\((.*?)\)\s*$')
tool_re = re.compile(r'TOOL:\s*([a-zA-Z_]+)\((.*)\)')

VALID_TOOL_PATTERN = re.compile(
    r'^TOOL:\s*(grep|count|peek)\("([^"]*)"(?:,"([^"]*)")?\)\s*$'
)

def parse_tool_call(text):
    """
    Only accept if the ENTIRE OUTPUT is ONE valid tool call line.
    Otherwise return None.
    """
    text = text.strip()
    # Must not contain newlines — only one line allowed
    if "\n" in text:
        return None

    m = VALID_TOOL_PATTERN.match(text)
    if not m:
        return None

    tool_name = m.group(1)
    arg1 = m.group(2)
    arg2 = m.group(3)

    if tool_name in ("grep", "count") and arg2 is not None:
        return None  # too many args

    if tool_name == "peek" and arg1 is None:
        return None  # missing second arg

    return tool_name, [arg for arg in (arg1, arg2) if arg is not None]



In [176]:
print(parse_tool_call('TOOL: peek("10000, 20")'))
print(parse_tool_call('TOOL: grep("Index=7777")'))

('peek', ['10000, 20'])
('grep', ['Index=7777'])


In [177]:
def tool_grep(pattern):
    if pattern == "Index=1234":
        return "LogEntry: Timestamp=2024-11-15, Index=1234, Value=OK"
    else:
        return f"LogEntry: Timestamp=2025-11-25, {pattern}, Value=OK"


def tool_peek(offset, length):
    return f"[fairytale {offset}:{length}]"

def tool_count(pattern):
    return f"[fake count for '{pattern}'] = 3"


TOOLS = {
    "grep": tool_grep,
    "peek": tool_peek,
    "count": tool_count,
}


In [167]:
SYSTEM_PROMPT = """
You are an RLM (Recursive Language Model) operating inside a controlled tool-use environment.

Your job is to extract information from a long document by using the available tools.
You MUST follow the rules EXACTLY.

====================================================
AVAILABLE TOOLS (STRICT SIGNATURES)
====================================================

You may call ONLY these tools, with EXACTLY these signatures:

1. grep(pattern: string) → string
   - Searches the external document for lines containing `pattern`.
   - Takes EXACTLY ONE string argument.

2. peek(offset: string, length: string) → string
   - Reads raw text from the external document.
   - Takes EXACTLY ONE string argument made of two integers seperated by a comma.

3. count(pattern: string) → string
   - Counts occurrences of `pattern`.
   - Takes EXACTLY ONE string argument.

You MUST NOT:
- Invent additional arguments.
- Omit required arguments.
- Change argument count.
- Change names.
- Combine tools.

STRICT ARGUMENT COUNT RULE:
A tool call with the wrong number of arguments is INVALID.

====================================================
FORMAT RULES FOR RESPONSES
====================================================

After every message (user query OR tool output), you MUST respond with exactly ONE of:

----------------------------------------------------
(1) A SINGLE TOOL CALL in this EXACT format:
----------------------------------------------------

TOOL: toolName("arg1, arg2")

Rules:
- Must begin with EXACTLY:  TOOL:
- No text before or after.
- No explanations.
- No additional prose.
- No JSON, brackets, tags, or role markers.
- You MUST include quotes around the argument.
- You MUST NOT output anything else.

Valid examples:
TOOL: grep("Index=1234")
TOOL: peek("200, 50")
TOOL: peek("2000, 50")
TOOL: count("ERROR")
TOOL: peek("20000, 50")

INVALID examples (meaning DO NOT DO THESE):
grep("Index=1234")
TOOL:grep("Index=1234")
TOOL: grep(Index=1234)
Tool: grep("Index=1234")
"TOOL: grep('Index=1234')"
<tool>grep("Index=1234")</tool>
TOOL: peek("20000","50")
TOOL: peek("20000")
TOOL: peek("20000","")


----------------------------------------------------
(2) A FINAL NATURAL LANGUAGE ANSWER
----------------------------------------------------

You may produce a FINAL ANSWER ONLY when:
- You already received the necessary tool output, AND
- You no longer need any additional tools.

Final answers MUST be plain English text with no tool calls.

Once you output a final answer, you MUST NOT call any tools again.

==============================
FINAL ANSWER COMPLETION RULES
==============================

A FINAL ANSWER must obey ALL of the following:

1. It MUST be pure natural language text.
2. It MUST NOT contain the words "TOOL:", "Tool:", "tool:", or any parentheses that look like tool calls.
3. It MUST NOT contain any quoted strings that resemble arguments.
4. It MUST NOT contain multiple lines describing actions.
5. It MUST end the response immediately — no additional text after the answer.

If you ever include a tool call-like structure after a natural language answer,
the response becomes INVALID.

If you produce a final answer, STOP immediately.

====================================================
STRICT HANDLING OF TOOL OUTPUT BLOCKS
====================================================

The Python environment will send tool results to you in this exact format:

[TOOL_OUTPUT_BEGIN]
...content...
[TOOL_OUTPUT_END]

This block:
- Contains evidence you MUST use.
- Is NOT a user message.
- MUST NOT be repeated.
- MUST NOT be modified.
- MUST NOT be ignored.
- MUST NOT trigger natural-language questions.
- MUST only lead to another tool call OR a final answer.

====================================================
INVALID MODEL OUTPUT POLICY
====================================================

Any output that does NOT start with EXACTLY:

TOOL:

and does NOT qualify as a FINAL ANSWER is INVALID.

INVALID outputs that SHOULD NOT BE DONE include:
- grep("pattern")
- "TOOL: grep("pattern")"
- TOOL_OUTPUT: ...
- Anything containing extra text with a tool call
- Anything that is NOT a tool call or final answer

====================================================
STOPPING RULES (VERY IMPORTANT AND STRICT)
====================================================

You MUST STOP and output a final answer when:
- The requested information has been found in a tool output.
- The question can be answered directly from the evidence.

Example:
User asks: "Find the line containing 'Index=1234'."

If grep("Index=1234") returns a line containing that text,
you MUST stop and output that line as a FINAL ANSWER.

You MUST NOT:
- Call grep again.
- Create new tool calls based on evidence.
- Try additional patterns.
- Loop forever.

====================================================
OBEY THESE RULES EXACTLY
====================================================

Failure to use the correct tool-call format,
failure to obey argument-count restrictions,
or failure to stop when evidence is sufficient
is considered INVALID.

=========================
STRICT NUMERIC HANDLING
=========================

When the user provides numeric values (like offsets, ranges, indices, lengths, byte positions, counts),
you MUST copy the numbers EXACTLY as given.

NEVER change, shorten, truncate, reinterpret, divide, or approximate numeric values.

Examples:
- If user says "offset 20000", you MUST use peek("20000, ..."), NOT peek("200, ...").
- If user says "length 300", you MUST use "300" exactly.
- If user says "Index=1234", you MUST preserve "1234".

If a number is not mentioned explicitly, you may choose a reasonable value,
BUT when a number *is* mentioned explicitly, you must use the exact number.


Begin.
"""
print(len(SYSTEM_PROMPT))
def format_chatml(messages):
    out = ""
    for msg in messages:
        role = msg["role"]
        content = msg["content"]
        out += f"<|im_start|>{role}\n{content}\n<|im_end|>\n"

    # IMPORTANT: open ONLY ONE assistant turn for the model to continue.
    out += "<|im_start|>assistant\n"
    return out
def append_tool_output(messages, tool_text):
    messages.append({
        "role": "assistant",
        "content": f"[TOOL_OUTPUT_BEGIN]\n{tool_text}\n[TOOL_OUTPUT_END]"
    })
def run_rlm_step(messages, model):
    prompt = format_chatml(messages)

    resp = client.chat.completions.create(
        model=model,
        messages=[],
        temperature=0,
        max_tokens=200,
        stop=["<|im_end|>"],  # important
        extra_body={"prompt": prompt}
    )

    text = resp.choices[0].message["content"].strip()
    return text

def run_model_step(messages, tokenizer, model):
    chat_text = format_chatml(messages)
    inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            eos_token_id=tokenizer.convert_tokens_to_ids("<|im_end|>")
        )

    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    decoded = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return decoded


def make_messages(user_query):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]


5760


In [165]:
def rlm_repl(user_query, tokenizer, model, max_steps=8):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for step in range(max_steps):
        print(f"\n=== MODEL STEP {step} ===")

        # Format chat
        chat_text = format_chatml(messages)
        inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)

        # Generate output
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False
            )

        generated = output[0][inputs["input_ids"].shape[1]:]
        response = tokenizer.decode(generated, skip_special_tokens=True).strip()

        print("MODEL RAW OUTPUT:", repr(response))

        # Try to parse a tool call
        parsed = parse_tool_call(response)

        if parsed is None:
          #print("Final Answer:", response)
          return response   # <-- STOP RUNNING


        # VALID TOOL CALL
        tool_name, args = parsed
        #print("Tool Requested:", tool_name, args)
        #print(tool_name in TOOLS)
        # If model output starts with TOOL:
        if tool_name in TOOLS:
            if tool_name == "peek":
                # args[0] is like "20000, 50"
                parts = args[0].split(",")
                if len(parts) != 2:
                    print("Malformed peek arguments.")
                    return "Model failed: malformed peek arguments."
                offset = parts[0].strip()
                length = parts[1].strip()
                tool_result = TOOLS["peek"](offset, length)
            else:
                tool_result = TOOLS[tool_name](*args)
            #print("Tool Output:", tool_result)

            # Append tool output block

            messages.append({
                "role": "assistant",
                "content": (
                    "[TOOL_OUTPUT_BEGIN]\n"
                    f"{tool_result}\n"
                    "[TOOL_OUTPUT_END]"
                )
            })

            continue

        # FINAL ANSWER
        #print("Final Answer:", response)
        return response

    # If all steps exhausted:
    return "Model failed to produce a valid response."


In [168]:
sid = rlm_repl("Find the line containing the text 'Index=1234'.",tokenizer,model)
print(sid)


=== MODEL STEP 0 ===
MODEL RAW OUTPUT: 'TOOL: grep("Index=1234")'

=== MODEL STEP 1 ===
MODEL RAW OUTPUT: "The line containing 'Index=1234' is:\nTimestamp=2024-11-15, Index=1234, Value=OK"
The line containing 'Index=1234' is:
Timestamp=2024-11-15, Index=1234, Value=OK


In [169]:
sid = rlm_repl("What text appears around character offset 20000?",tokenizer,model)
print(sid)


=== MODEL STEP 0 ===
MODEL RAW OUTPUT: 'TOOL: peek("20000, 100")'

=== MODEL STEP 1 ===
MODEL RAW OUTPUT: 'The text around character offset 20000 is "fairytale".'
The text around character offset 20000 is "fairytale".


In [170]:
rlm_repl("Tell me a joke", tokenizer, model)


=== MODEL STEP 0 ===
MODEL RAW OUTPUT: "Why don't scientists trust atoms? Because they make up everything."


"Why don't scientists trust atoms? Because they make up everything."

In [179]:
start = time.time()
rlm_repl("Where is Index=7777 mentioned?", tokenizer, model)
print("Elapsed:", time.time() - start)


=== MODEL STEP 0 ===
MODEL RAW OUTPUT: 'TOOL: grep("Index=7777")'

=== MODEL STEP 1 ===
MODEL RAW OUTPUT: 'The line mentioning Index=7777 is at offset 20000. \n\n[TOOL: peek("20000, 50")]'
Elapsed: 5.343576431274414
